In [1]:
!hdfs dfs -mkdir -p /user/ibrahim/ecommerce/raw
!hdfs dfs -mkdir -p /user/ibrahim/ecommerce/processed

!hdfs dfs -ls /user/ibrahim/ecommerce

Found 2 items
drwxr-xr-x   - ibrahim supergroup          0 2026-09-09 16:23 /user/ibrahim/ecommerce/processed
drwxr-xr-x   - ibrahim supergroup          0 2026-09-09 16:22 /user/ibrahim/ecommerce/raw


In [3]:
!hdfs dfs -put transaksi_magelang.csv /user/ibrahim/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/ibrahim/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/ibrahim/ecommerce/raw/

!hdfs dfs -ls -h /user/ibrahim/ecommerce/rawimport pandas as pd

Found 3 items
-rw-r--r--   1 ibrahim supergroup     12.0 K 2026-09-09 16:29 /user/ibrahim/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 ibrahim supergroup     11.9 K 2026-09-09 16:30 /user/ibrahim/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 ibrahim supergroup     12.4 K 2026-09-09 16:29 /user/ibrahim/ecommerce/raw/transaksi_yogyakarta.csv


In [7]:
!pip install hdfs

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hdfs: filename=hdfs-2.7.3-py3-none-any.whl size=34431 sha256=b34d74c3822cc8e739b7e3a5af846abc41cde2dc8d36252db450574c873a059c
  Stored in directory: /home/ibrahim/.cache/pip/wheels/b9/1d/dc/eb0833be25464c359903d356c4204721c6a672c26ff164cdc3
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13783 sha256=ebcd0cf8b3162c8dfbce4bb11339d222da8d2d80cf72579b6024268b54b0fd61
  Stored in directory: /home/ibrahim/.cache/pip/wheels/1a/b0/8c/4b75c4116c31f83c8f9f047231251e13cc74481cca4a78a9ce
Successfully built hdfs docopt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [hdfs]


In [8]:
import pandas as pd
from hdfs import InsecureClient


client = InsecureClient('http://localhost:9870', user='ibrahim')

files = [
    '/user/ibrahim/ecommerce/raw/transaksi_magelang.csv',
    '/user/ibrahim/ecommerce/raw/transaksi_yogyakarta.csv',
    '/user/ibrahim/ecommerce/raw/transaksi_semarang.csv'
]

dfs = []
for file in files:
    with client.read(file) as reader:
        dfs.append(pd.read_csv(reader))


df_gabungan = pd.concat(dfs, ignore_index=True)


print(df_gabungan['kota'].value_counts())

kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


In [9]:
df_gabungan['total_pendapatan'] = df_gabungan['unit_terjual'] * df_gabungan['harga_satuan']


ringkasan_df = df_gabungan.groupby(['kota', 'kategori'])['total_pendapatan'].sum().reset_index()


df_gabungan.to_csv('data_gabungan_bersih.csv', index=False)
ringkasan_df.to_csv('ringkasan_kota_kategori.csv', index=False)


!hdfs dfs -put -f data_gabungan_bersih.csv /user/ibrahim/ecommerce/processed/
!hdfs dfs -put -f ringkasan_kota_kategori.csv /user/ibrahim/ecommerce/processed/


!hdfs dfs -ls -h /user/ibrahim/ecommerce/processed

Found 2 items
-rw-r--r--   1 ibrahim supergroup     40.3 K 2026-09-09 16:46 /user/ibrahim/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 ibrahim supergroup        530 2026-09-09 16:47 /user/ibrahim/ecommerce/processed/ringkasan_kota_kategori.csv


## E. Dokumentasi dan Refleksi

![Struktur Folder HDFS Ecommerce](hdfs.png)

### 2. Tulisan Reflektif

Pemisahan antara direktori raw (data mentah) dan processed (data olahan) dalam HDFS merupakan implementasi praktik terbaik (best practice) dalam arsitektur manajemen Big Data. Pendekatan ini menjaga prinsip data immutability, di mana data asli yang masuk dari berbagai sumber sistem (ingestion) tetap utuh tanpa risiko terubah atau terhapus secara tidak sengaja akibat proses transformasi. Data mentah berfungsi sebagai single source of truth yang dapat digunakan kembali jika di masa mendatang terdapat perubahan logika bisnis, koreksi bug pada pipelines, atau kebutuhan analisis model baru tanpa perlu melakukan pengumpulan data ulang dari sistem sumber.

Selain itu, pemisahan direktori meningkatkan efisiensi tata kelola data (data governance) dan keamanan. Pengelola sistem dapat menerapkan kontrol akses (ACL) yang lebih ketat pada folder raw agar hanya proses ingestion terotorisasi yang dapat menulis data. Sementara itu, tim data analyst atau data scientist dapat diberikan akses mandiri ke folder processed untuk kebutuhan query dan pembuatan laporan. Struktur ini juga mempermudah pemantauan siklus hidup data (lifecycle management) serta optimasi storage, misalnya dengan menerapkan skema kompresi atau format penyimpanan yang berbeda pada folder processed tanpa mengganggu ketersediaan berkas CSV mentah aslinya.